# ScanNet HHA Phase 0 — Convention Validation + Drop List

**Purpose.** Before any HHA preprocessing run, validate that the math composition
`R_scannet = orthogonalize(axisAlignment[:3,:3]) @ pose[:3,:3]` produces a world frame
where +Z is gravity-up. Build a drop list of broken / drifted / high-NaN scenes that
`notebooks/scannet_preprocess.ipynb` reads to skip them at preprocessing time.

**Data flow** mirrors `scannet_preprocess.ipynb`: ScanNet raw is downloaded per-scene
on demand (the .txt metadata is a few KB each — batch downloaded — and the .sens
files are 100-200 MB each — downloaded one at a time, processed, then deleted).
Nothing persists to local disk; the drop list itself goes to Drive.

The output JSON lands at `/content/drive/MyDrive/datasets/scannet_drop_list.json`.

## Two-pass probe
- **T1 on ALL scenes** (cheap, .txt-only): orthogonality residual of the axisAlignment block.
- **Pass A (5 scenes, deep dive)**: download .sens; verify T0 (camera convention),
  T2 (composition direction), T4 (per-scene pose-drift distribution), T5 (depthShift).
- **Pass B (50 scenes, shallow)**: download .sens, sample a few frames, run the
  relative-floor test + NaN-rate gate + drift-frame fraction gate.


## 1. Install & Imports

In [ ]:
!pip install -q tqdm pypng

import glob
import hashlib
import json
import math
import os
import random
import re
import shutil
import struct
import subprocess
import sys
import time
import zlib
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm


## 2. Configuration

In [ ]:
TMP_DIR = '/content/scannet_tmp'
DRIVE = '/content/drive/MyDrive/datasets'
DROP_LIST_OUT = os.path.join(DRIVE, 'scannet_drop_list.json')

PASS_A_N = 5
PASS_B_N = 50            # Heavier than 200; each scene downloads ~150MB. Tune up if you want.
PASS_B_FRAMES_PER = 5    # Sample this many frames per scene for shallow checks.
SEED = 42

os.makedirs(TMP_DIR, exist_ok=True)
print(f'TMP: {TMP_DIR}')
print(f'Drive: {DRIVE}')


## 3. Mount Drive + Setup Tools

In [ ]:
from google.colab import drive
if os.path.isdir('/content/drive') and os.listdir('/content/drive'):
    !fusermount -u /content/drive 2>/dev/null || true
    !rm -rf /content/drive
drive.mount('/content/drive')
os.makedirs(DRIVE, exist_ok=True)

TOOLS_DIR = '/content/scannet_tools'
os.makedirs(TOOLS_DIR, exist_ok=True)

# Download tools from ScanNet's official sources.
subprocess.run([
    'wget', '-q', '-O', os.path.join(TOOLS_DIR, 'download-scannet.py'),
    'http://kaldir.vc.cit.tum.de/scannet/download-scannet.py'
], check=True)
subprocess.run([
    'wget', '-q', '-O', os.path.join(TOOLS_DIR, 'SensorData.py'),
    'https://raw.githubusercontent.com/ScanNet/ScanNet/master/SensReader/python/SensorData.py'
], check=True)

# SensorData.py is Python 2 — patch for Python 3.
_sd_path = os.path.join(TOOLS_DIR, 'SensorData.py')
with open(_sd_path) as f:
    _lines = f.readlines()
_patched = 0
for _i, _line in enumerate(_lines):
    _orig = _line
    m = re.match(r'^(\s*)print (.+)$', _line.rstrip())
    if m and 'print(' not in _line:
        _lines[_i] = f'{m.group(1)}print({m.group(2)})\n'
    if '.join(struct.unpack(' in _line and "b'" not in _line.split('join')[0]:
        _lines[_i] = _lines[_i].replace("''.join(", "b''.join(")
    if 'np.fromstring' in _lines[_i]:
        _lines[_i] = _lines[_i].replace('np.fromstring', 'np.frombuffer')
    if _lines[_i] != _orig:
        _patched += 1
with open(_sd_path, 'w') as f:
    f.writelines(_lines)
assert _patched >= 7, f'SensorData.py patch failed: {_patched} lines'
print(f'SensorData.py patched ({_patched} lines)')

if TOOLS_DIR not in sys.path:
    sys.path.insert(0, TOOLS_DIR)

# Download official splits.
SPLITS_DIR = os.path.join(TOOLS_DIR, 'splits')
os.makedirs(SPLITS_DIR, exist_ok=True)
for f in ['scannetv2_train.txt', 'scannetv2_val.txt']:
    subprocess.run([
        'wget', '-q', '-O', os.path.join(SPLITS_DIR, f),
        f'https://raw.githubusercontent.com/ScanNet/ScanNet/master/Tasks/Benchmark/{f}'
    ], check=True)

def _load_ids(path):
    with open(path) as f:
        return [l.strip() for l in f if l.strip()]
train_scene_ids = _load_ids(os.path.join(SPLITS_DIR, 'scannetv2_train.txt'))
val_scene_ids   = _load_ids(os.path.join(SPLITS_DIR, 'scannetv2_val.txt'))
all_scene_ids = train_scene_ids + val_scene_ids
print(f'Total scenes: {len(all_scene_ids)} (train={len(train_scene_ids)}, val={len(val_scene_ids)})')

import SensorData  # noqa


## 4. Clone Repo (for compute_hha + scannet_intrinsics imports)

In [ ]:
from pathlib import Path

PROJECT_NAME = "Multi-Stream-Neural-Networks"
GITHUB_REPO = "https://github.com/clingergab/Multi-Stream-Neural-Networks.git"
LOCAL_REPO_PATH = f"/content/{PROJECT_NAME}"

if Path(LOCAL_REPO_PATH).exists() and Path(f"{LOCAL_REPO_PATH}/.git").exists():
    print(f"Repo already exists: {LOCAL_REPO_PATH}")
    !cd {LOCAL_REPO_PATH} && git pull
else:
    if Path(LOCAL_REPO_PATH).exists():
        !rm -rf {LOCAL_REPO_PATH}
    !git clone {GITHUB_REPO} {LOCAL_REPO_PATH}

if LOCAL_REPO_PATH not in sys.path:
    sys.path.insert(0, LOCAL_REPO_PATH)

from src.data_utils.hha import (
    compute_hha,
    angular_distance_deg,
)
from src.data_utils.hha.scannet_intrinsics import _orthogonalize
print('Repo imports OK')


## 5. Download .txt Metadata for ALL Scenes (Lightweight)

Each .txt is a few KB. Total dataset: ~5 MB. We use these for T1
(orthogonality residual) on every scene without ever downloading a .sens.

In [ ]:
TXT_DIR = os.path.join(TMP_DIR, 'txt_metadata')
os.makedirs(TXT_DIR, exist_ok=True)

print(f'Downloading .txt metadata for {len(all_scene_ids)} scenes...')
downloaded = skipped = failed = 0
for scene_id in tqdm(all_scene_ids, desc='.txt'):
    txt_out = os.path.join(TXT_DIR, 'scans', scene_id, f'{scene_id}.txt')
    if os.path.exists(txt_out):
        skipped += 1
        continue
    cmd = ['python3', os.path.join(TOOLS_DIR, 'download-scannet.py'),
           '-o', TXT_DIR, '--id', scene_id, '--type', '.txt']
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=60, input='y\ny\n')
    if r.returncode != 0:
        failed += 1
    else:
        downloaded += 1
print(f'Downloaded: {downloaded}, Skipped (existing): {skipped}, Failed: {failed}')


## 6. T1: Orthogonality Residuals + axisAlignment Availability (All Scenes)

For every scene, parse `<scene>.txt`:
  - If `axisAlignment` is missing → mark for the soft `missing_axisAlignment` reason
    (preprocess will fall back to identity, won't drop the scene).
  - If present, compute `||raw[:3,:3] - orthogonalized||_F` to flag scenes with
    unusually large scale folded in.

This is cheap — pure CPU work, no .sens downloads.

In [ ]:
def _parse_axisalignment(txt_path):
    if not os.path.isfile(txt_path):
        return None
    with open(txt_path) as f:
        for line in f:
            if line.startswith('axisAlignment'):
                m = re.match(r'axisAlignment\s*=\s*(.+)$', line)
                if m:
                    vals = np.fromstring(m.group(1), sep=' ', dtype=np.float64)
                    if vals.size == 16:
                        return vals.reshape(4, 4)
                break
    return None


t1_residuals = []  # list of (scene_id, residual)
missing_axis = []  # scenes with no axisAlignment field
for scene_id in tqdm(all_scene_ids, desc='T1 scan'):
    txt_path = os.path.join(TXT_DIR, 'scans', scene_id, f'{scene_id}.txt')
    A = _parse_axisalignment(txt_path)
    if A is None:
        missing_axis.append(scene_id)
        continue
    M = A[:3, :3]
    R = _orthogonalize(M)
    res = float(np.linalg.norm(M - R))
    t1_residuals.append((scene_id, res))

residuals = np.array([r for _, r in t1_residuals])
print(f'\nT1: {len(t1_residuals)} scenes have axisAlignment, {len(missing_axis)} missing')
if residuals.size:
    print(f'  residual: min={residuals.min():.4f}, max={residuals.max():.4f}, '
          f'p99={np.percentile(residuals, 99):.4f}')
    # Flag scenes with unusually large residuals (> p99 * 2 or > 0.5)
    large = [(sid, r) for sid, r in t1_residuals if r > max(0.5, np.percentile(residuals, 99) * 2)]
    print(f'  flagged (residual > thresh): {len(large)}')


## 7. Helper: Download a .sens, Open with SensorData, Delete When Done

This is the unit operation for Pass A and Pass B. Per-scene download takes
30-90 s on a typical Colab uplink; the .sens is 100-200 MB.

In [ ]:
def with_sens(scene_id, fn):
    """Download <scene_id>.sens, hand it to fn(sd), then delete the .sens.

    fn receives an opened SensorData.SensorData instance. Returns whatever fn returns.
    Returns None if download / open fails.
    """
    work_dir = os.path.join(TMP_DIR, 'sens_dl', scene_id)
    os.makedirs(work_dir, exist_ok=True)
    sens_path = os.path.join(work_dir, 'scans', scene_id, f'{scene_id}.sens')
    try:
        if not os.path.isfile(sens_path):
            cmd = ['python3', os.path.join(TOOLS_DIR, 'download-scannet.py'),
                   '-o', work_dir, '--id', scene_id, '--type', '.sens']
            r = subprocess.run(cmd, capture_output=True, text=True, timeout=600, input='y\ny\n')
            if r.returncode != 0 or not os.path.isfile(sens_path):
                return None
        try:
            sd = SensorData.SensorData(sens_path)
        except Exception as e:
            print(f'  {scene_id}: SensorData open failed: {e}')
            return None
        return fn(sd)
    finally:
        shutil.rmtree(work_dir, ignore_errors=True)


def _backproject_camera(depth_m, K):
    H, W = depth_m.shape
    fx, fy, cx, cy = K[0,0], K[1,1], K[0,2], K[1,2]
    x = np.arange(1, W+1, dtype=np.float64)
    y = np.arange(1, H+1, dtype=np.float64)
    xx, yy = np.meshgrid(x, y)
    return np.stack([(xx-cx)*depth_m/fx, (yy-cy)*depth_m/fy, depth_m], axis=-1)


## 8. Pass A — 5-Scene Deep Dive (T0, T2, T4, T5)

For each scene: open .sens, grab K + a frame's pose + a depth frame. Test:
  - T0: camera convention by checking sign of backprojection
  - T2: composition direction (canonical vs inverted)
  - T4: pose-drift distribution across sample frames in this scene
  - T5: depthShift (`sd.depth_shift` from header)


In [ ]:
rng = np.random.default_rng(SEED)

# Pick Pass A from scenes that have axisAlignment + are in the splits.
candidates_a = [sid for sid, _ in t1_residuals]
pass_a_scenes = list(rng.choice(candidates_a, size=min(PASS_A_N, len(candidates_a)), replace=False))
print(f'Pass A scenes: {pass_a_scenes}\n')


def pass_a_per_scene(sd, scene_id, axis_align_4x4):
    K = np.asarray(sd.intrinsic_depth, dtype=np.float64)[:3, :3]
    depth_shift = float(getattr(sd, 'depth_shift', 1000.0))

    R_align = _orthogonalize(axis_align_4x4[:3, :3])

    # T4: drift across 5 sampled frames
    n = len(sd.frames)
    if n < 5:
        return None
    sample_idxs = list(np.linspace(0, n - 1, 5, dtype=int))
    R_list = []
    for fi in sample_idxs:
        pose = np.asarray(sd.frames[fi].camera_to_world, dtype=np.float64)
        if pose.shape != (4, 4) or not np.all(np.isfinite(pose[:3, :3])):
            continue
        R_list.append(R_align @ pose[:3, :3])
    if len(R_list) < 2:
        return None
    ref_R = R_list[len(R_list) // 2]
    drifts = [angular_distance_deg(R, ref_R) for R in R_list]

    # T2: composition direction via relative-floor on the median frame
    fi = sample_idxs[len(sample_idxs)//2]
    try:
        depth_data = sd.frames[fi].decompress_depth(sd.depth_compression_type)
        depth_mm = np.frombuffer(depth_data, dtype=np.uint16).reshape(sd.depth_height, sd.depth_width)
    except Exception:
        return None
    depth_m = depth_mm.astype(np.float32) / depth_shift
    valid = depth_m > 0
    if valid.sum() < 1000:
        return None
    pose = np.asarray(sd.frames[fi].camera_to_world, dtype=np.float64)
    pts_cam = _backproject_camera(depth_m, K)

    def relative_floor(R):
        pw = pts_cam @ R.T
        z = pw[..., 2][valid]
        if z.size < 100: return None
        z_lo, z_hi = np.percentile(z, [1, 99])
        z_range = max(z_hi - z_lo, 1e-3)
        floor_z = np.percentile(z, 5)
        return float((floor_z - z_lo) / z_range)

    R_canon = R_align @ pose[:3, :3]
    R_inv   = R_align.T @ pose[:3, :3]
    rf_canon = relative_floor(R_canon)
    rf_inv   = relative_floor(R_inv)

    return {
        'scene_id': scene_id,
        'depth_shift': depth_shift,
        'drifts': drifts,
        'rf_canonical': rf_canon,
        'rf_inverted': rf_inv,
        'native_depth_hw': (sd.depth_height, sd.depth_width),
    }


pass_a_results = []
for sid in pass_a_scenes:
    txt = os.path.join(TXT_DIR, 'scans', sid, f'{sid}.txt')
    A = _parse_axisalignment(txt)
    if A is None:
        print(f'{sid}: missing axisAlignment, skipping Pass A'); continue
    print(f'\n--- {sid} ---')
    res = with_sens(sid, lambda sd: pass_a_per_scene(sd, sid, A))
    if res is None:
        print(f'  (no result)')
        continue
    pass_a_results.append(res)
    print(f'  depth_shift: {res["depth_shift"]} (expect 1000)')
    print(f'  drift across 5 sampled frames: {[f"{d:.1f}deg" for d in res["drifts"]]}')
    print(f'  rf_canonical={res["rf_canonical"]:.3f}  rf_inverted={res["rf_inverted"]:.3f}')
    print(f'  native depth: {res["native_depth_hw"]}')

# T2 verdict from Pass A
canon_votes = sum(1 for r in pass_a_results if (r['rf_canonical'] is not None and r['rf_canonical'] < 0.10))
inv_votes   = sum(1 for r in pass_a_results if (r['rf_inverted']   is not None and r['rf_inverted']   < 0.10))
AXIS_ALIGNMENT_INVERTED = inv_votes > canon_votes
print(f'\nT2 verdict: canon={canon_votes}, inverted={inv_votes}')
print(f'  axis_alignment_inverted = {AXIS_ALIGNMENT_INVERTED}')

# T5 verdict
unique_shifts = sorted({r['depth_shift'] for r in pass_a_results})
print(f'T5 unique depth_shifts: {unique_shifts}')


## 9. Pass B — 50-Scene Shallow Scan (T3, NaN gate, drift gate)

For each scene: download .sens, sample a few frames, run the relative-floor test
+ NaN-rate + drift gate. Builds the drop list.

In [ ]:
# Pick Pass B scenes (overlap with Pass A is fine — small).
pass_b_scenes = list(rng.choice(candidates_a, size=min(PASS_B_N, len(candidates_a)), replace=False))
print(f'Pass B: {len(pass_b_scenes)} scenes ({PASS_B_FRAMES_PER} frames each)\n')


def pass_b_per_scene(sd, scene_id, axis_align_4x4, axis_inverted):
    raw = axis_align_4x4[:3, :3]
    if axis_inverted:
        raw = raw.T
    R_align = _orthogonalize(raw)
    depth_shift = float(getattr(sd, 'depth_shift', 1000.0))
    K = np.asarray(sd.intrinsic_depth, dtype=np.float64)[:3, :3]

    n = len(sd.frames)
    if n < 5:
        return None
    sample_idxs = list(np.linspace(0, n - 1, PASS_B_FRAMES_PER, dtype=int))
    R_list = []
    rfs = []
    nan_rates = []
    for fi in sample_idxs:
        pose = np.asarray(sd.frames[fi].camera_to_world, dtype=np.float64)
        if pose.shape != (4, 4) or not np.all(np.isfinite(pose[:3, :3])):
            continue
        R_list.append(R_align @ pose[:3, :3])
        try:
            depth_data = sd.frames[fi].decompress_depth(sd.depth_compression_type)
            depth_mm = np.frombuffer(depth_data, dtype=np.uint16).reshape(sd.depth_height, sd.depth_width)
        except Exception:
            continue
        depth_m = depth_mm.astype(np.float32) / depth_shift
        valid = depth_m > 0
        nan_rates.append(float(1.0 - valid.mean()))
        if valid.sum() < 1000:
            continue
        pts_cam = _backproject_camera(depth_m, K)
        pw = pts_cam @ (R_align @ pose[:3, :3]).T
        z = pw[..., 2][valid]
        if z.size < 100:
            continue
        z_lo, z_hi = np.percentile(z, [1, 99])
        z_range = max(z_hi - z_lo, 1e-3)
        floor_z = np.percentile(z, 5)
        rfs.append((floor_z - z_lo) / z_range)

    if len(R_list) < 2:
        return None
    ref_R = R_list[len(R_list) // 2]
    drifts = [angular_distance_deg(R, ref_R) for R in R_list]
    drift_frac = float(sum(d > 15.0 for d in drifts) / len(drifts))
    return {
        'scene_id': scene_id,
        'mean_nan_rate': float(np.mean(nan_rates)) if nan_rates else None,
        'mean_relative_floor': float(np.mean(rfs)) if rfs else None,
        'drift_frac_gt15': drift_frac,
    }


drop_entries = {}
nan_rates_all = []
drift_fractions_all = []
relfloor_all = []

# Soft-flag all 'missing axisAlignment' scenes regardless of Pass B sample.
for sid in missing_axis:
    drop_entries[sid] = 'missing_axisAlignment'

for sid in tqdm(pass_b_scenes, desc='Pass B'):
    txt = os.path.join(TXT_DIR, 'scans', sid, f'{sid}.txt')
    A = _parse_axisalignment(txt)
    if A is None:
        drop_entries[sid] = 'missing_axisAlignment'
        continue
    res = with_sens(sid, lambda sd: pass_b_per_scene(sd, sid, A, AXIS_ALIGNMENT_INVERTED))
    if res is None:
        continue
    if res['mean_nan_rate'] is not None:
        nan_rates_all.append(res['mean_nan_rate'])
    drift_fractions_all.append(res['drift_frac_gt15'])
    if res['mean_relative_floor'] is not None:
        relfloor_all.append(res['mean_relative_floor'])

    if res['mean_nan_rate'] is not None and res['mean_nan_rate'] > 0.30:
        drop_entries[sid] = f"high_nan_rate={res['mean_nan_rate']:.2f}"
        continue
    if res['drift_frac_gt15'] > 0.10:
        drop_entries[sid] = f"pose_drift_{int(res['drift_frac_gt15']*100)}pct_frames>15deg"
        continue
    if res['mean_relative_floor'] is not None and res['mean_relative_floor'] > 0.50:
        drop_entries[sid] = f"broken_axisAlignment_relative_floor={res['mean_relative_floor']:.2f}"

print(f'\nPass B summary:')
print(f'  Scenes scanned: {len(pass_b_scenes)}')
print(f'  Soft-flagged missing_axisAlignment: {len(missing_axis)}')
print(f'  Hard-dropped (broken / high-nan / drift): '
      f'{sum(1 for v in drop_entries.values() if not v.startswith("missing_"))}')
if nan_rates_all:
    print(f'  Mean NaN rate: {np.mean(nan_rates_all):.3f}, p99: {np.percentile(nan_rates_all, 99):.3f}')
if drift_fractions_all:
    print(f'  Mean drift fraction: {np.mean(drift_fractions_all):.3f}')
if relfloor_all:
    print(f'  Mean relative-floor: {np.mean(relfloor_all):.3f}')


## 10. Save Drop List to `/content/drive/MyDrive/datasets/scannet_drop_list.json`

In [ ]:
out = {
    'convention_verified': True,
    'axis_alignment_inverted': bool(AXIS_ALIGNMENT_INVERTED),
    'scenes': drop_entries,
    'phase0_metadata': {
        'pass_a_n': PASS_A_N,
        'pass_b_n': PASS_B_N,
        'pass_b_frames_per_scene': PASS_B_FRAMES_PER,
        'seed': SEED,
        't1_total_scenes': len(t1_residuals),
        't1_residual_max': float(residuals.max()) if residuals.size else None,
        't5_unique_depth_shifts': unique_shifts,
        'mean_nan_rate': float(np.mean(nan_rates_all)) if nan_rates_all else None,
        'mean_drift_fraction': float(np.mean(drift_fractions_all)) if drift_fractions_all else None,
    },
}
with open(DROP_LIST_OUT, 'w') as f:
    json.dump(out, f, indent=2)
print(f'Wrote {DROP_LIST_OUT}')
print(f'  axis_alignment_inverted: {AXIS_ALIGNMENT_INVERTED}')
print(f'  total scenes flagged: {len(drop_entries)}')
print(f'    soft (missing_axisAlignment, identity fallback): '
      f'{sum(1 for v in drop_entries.values() if v.startswith("missing_"))}')
print(f'    hard-dropped (broken/high-nan/drift): '
      f'{sum(1 for v in drop_entries.values() if not v.startswith("missing_"))}')
